# PySpark

This lectures provides an overview of PySpark.

There are two ways to run the content in this server: connect to a remote server or run via a local instance. These options can be controlled via the variable `USE_REMOTE`. 

If you want to run your code directly via a .py file, you would use the following command:

```python
spark-submit --master spark://your-server-ip:7077 your_script.py
```

In [129]:
# If connecting to a remote server
# !pip install "pyspark[connect]"==4.0.1 # very lightweight

# If connecting to a local, dev, instance
#!pip install pyspark==4.0.1 # bigger download

In [130]:
import getpass
from pyspark.sql import SparkSession
import pandas as pd

In [131]:
# Remote server
MASTER_IP = "51.222.140.217"
REMOTE_SERVER = f"sc://{MASTER_IP}/15002"

USE_REMOTE = False 

In [132]:
f"Dashboard available at http://{MASTER_IP}:8080/" if USE_REMOTE else f"Dashboard available at http://localhost:4040/"

'Dashboard available at http://localhost:4040/'

In [133]:
# Note that spark.sql.ansi.enabled is set to false to allow 'Pandas on Spark' to work correctly later in the lecture
if USE_REMOTE:
    spark = SparkSession.builder.remote(MASTER_IP).appName(f"job_id_{getpass.getuser()}").config("spark.sql.ansi.enabled", "false").getOrCreate()
else:
    spark = SparkSession.builder.master("local[*]").appName(f"job_id_{getpass.getuser()}").config("spark.sql.ansi.enabled", "false") .getOrCreate()

In [134]:
# Test it
df = spark.range(1000)
print(df.count())

1000


Notes:

Remote `connect server` for Spark may be started via `/opt/spark/sbin/start-connect-server.sh`

If no remote installation exists, the following script will install it:
```bash
wget https://archive.apache.org/dist/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz
tar -zxf spark-4.0.1-bin-hadoop3.tgz 
sudo mv spark-4.0.1-bin-hadoop3 /opt/spark
rm spark-4.0.1-bin-hadoop3.tgz
```

In [135]:
df.explain()

== Physical Plan ==
*(1) Range (0, 1000, step=1, splits=16)




## Spark distributes data

Read via Pandas for comparison

In [136]:
%%time
trades_df = pd.read_csv("../../datasets/market_data/trades_2025-09-10_AAPLMSFT_sorted.csv.gz")

CPU times: user 277 ms, sys: 15.8 ms, total: 293 ms
Wall time: 292 ms


In [137]:
type(trades_df)

pandas.core.frame.DataFrame

In [138]:
trades_df.columns

Index(['ticker', 'price', 'size', 'sip_timestamp'], dtype='object')

In [139]:
%%time
len(trades_df)

CPU times: user 4 μs, sys: 1e+03 ns, total: 5 μs
Wall time: 4.77 μs


1504365

In [140]:
del trades_df

Now read via Pyspark

In [141]:
%%time
trades_df = spark.read.csv("../../datasets/market_data/trades_2025-09-10_AAPLMSFT_sorted.csv.gz", header=True, inferSchema=True)

CPU times: user 1.18 ms, sys: 939 μs, total: 2.12 ms
Wall time: 988 ms


25/12/02 20:14:54 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:700)
	at org.apache.spark.storage.BlockManagerMasterE

In [142]:
type(trades_df)

pyspark.sql.classic.dataframe.DataFrame

In [143]:
trades_df.columns

['ticker', 'price', 'size', 'sip_timestamp']

In [144]:
%%time
trades_df.count()

CPU times: user 295 μs, sys: 589 μs, total: 884 μs
Wall time: 224 ms


1504365

In [145]:
%%time
trades_df.show(5)

+------+------+----+-------------------+
|ticker| price|size|      sip_timestamp|
+------+------+----+-------------------+
|  MSFT|499.51| 217|1757491200016438411|
|  MSFT|498.45|   2|1757491200033420124|
|  AAPL|234.03|   1|1757491200037656528|
|  AAPL|234.14|  19|1757491200037664512|
|  AAPL| 234.0|   5|1757491200038141123|
+------+------+----+-------------------+
only showing top 5 rows
CPU times: user 362 μs, sys: 642 μs, total: 1 ms
Wall time: 17.2 ms


Note that Spark data is generally distributed and operations on it are also distributed

In [146]:
trades_df.rdd.getNumPartitions()

1

#### Pick specific columns

In [147]:
trades_df.select('ticker', 'price', 'size').show(5)

+------+------+----+
|ticker| price|size|
+------+------+----+
|  MSFT|499.51| 217|
|  MSFT|498.45|   2|
|  AAPL|234.03|   1|
|  AAPL|234.14|  19|
|  AAPL| 234.0|   5|
+------+------+----+
only showing top 5 rows


#### Pick rows

In [148]:
trades_df.filter(trades_df.size > 200).show(5)

+------+------+----+-------------------+
|ticker| price|size|      sip_timestamp|
+------+------+----+-------------------+
|  MSFT|499.51| 217|1757491200016438411|
|  MSFT|499.51| 240|1757491200069918958|
|  MSFT| 500.0| 385|1757491200138873327|
|  MSFT| 500.0| 298|1757491248775896025|
|  AAPL|234.22| 240|1757491271106714912|
+------+------+----+-------------------+
only showing top 5 rows


#### Many operations in Spark are **lazy**, they don't execute until they are actually needed

This allows Spark to look at several operatoins together and possibly optimize them  
Notice the difference in execution speed

In [149]:
%%time
trades_df.groupby('ticker').mean()

CPU times: user 664 μs, sys: 848 μs, total: 1.51 ms
Wall time: 3.75 ms


DataFrame[ticker: string, avg(price): double, avg(size): double, avg(sip_timestamp): double]

In [150]:
%%time
trades_df.groupby('ticker').mean().show(5)

+------+-----------------+-----------------+--------------------+
|ticker|       avg(price)|        avg(size)|  avg(sip_timestamp)|
+------+-----------------+-----------------+--------------------+
|  AAPL|228.0181511970178| 89.6477142325466|1.757520113714569...|
|  MSFT| 500.529295227757|51.74418151924823|1.757519825771087...|
+------+-----------------+-----------------+--------------------+

CPU times: user 1.38 ms, sys: 1.03 ms, total: 2.41 ms
Wall time: 1.06 s


Generally speaking, function calls, such as `filter` or `select` don't actually transform the data! Only when data is actually demanded, by functions such as `show` or `collect`, does Pyspark optimize the built up commands and executes them.

#### Spark DataFrames are built on top of RDD (Resilient Distributed Datasets)

In [151]:
type(trades_df.rdd)

pyspark.core.rdd.RDD

An RDD is assumed to be distributed by default. It is a low level datastructure which is no longer used directly.

### SQL is built into Spark!

In [152]:
trades_df.createOrReplaceTempView("trades_df")

In [153]:
spark.sql("SELECT ticker, mean(price), mean(size) from trades_df group by ticker").show()

+------+-----------------+-----------------+
|ticker|      mean(price)|       mean(size)|
+------+-----------------+-----------------+
|  AAPL|228.0181511970178| 89.6477142325466|
|  MSFT| 500.529295227757|51.74418151924823|
+------+-----------------+-----------------+



### ... actually Pandas is also built into Spark
(although it is not as featureful as Spark's built-in Dataframes)

In [154]:
import pyspark.pandas as ps

In [155]:
trades_ds = ps.read_csv("../../datasets/market_data/trades_2025-09-10_AAPLMSFT_sorted.csv.gz")
trades_ds.head()

/opt/anaconda3/envs/finm/lib/python3.13/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,ticker,price,size,sip_timestamp
0,MSFT,499.51,217,1757491200016438411
1,MSFT,498.45,2,1757491200033420124
2,AAPL,234.03,1,1757491200037656528
3,AAPL,234.14,19,1757491200037664512
4,AAPL,234.00,5,1757491200038141123


In [156]:
type(trades_ds)

pyspark.pandas.frame.DataFrame

In [157]:
trades_ds.ticker.value_counts()

ticker
AAPL    1031776
MSFT     472589
Name: count, dtype: int64

## Machine learning in PySpark
Think of this as distributed scikit-learn!

Note that PySpark has has a set of libraries under `pyspark.mllib.*` and `pyspark.ml.*`. The _mllib_ set of packages are in maintenance mode. They were designed to work with RDDs, a predecessor to Dataframes. 

In [158]:
import seaborn as sns
titanic_df = sns.load_dataset('titanic')

#only keep numeric columns (for simplicity)
titanic_onlynum_df = titanic_df.drop(['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive'], axis=1)

#remove na, nan, etc.
titanic_onlynum_noempty_df = titanic_onlynum_df.dropna()
titanic_df = spark.createDataFrame(titanic_onlynum_noempty_df)

In [159]:
titanic_df.show(5)

+--------+------+----+-----+-----+-------+----------+-----+
|survived|pclass| age|sibsp|parch|   fare|adult_male|alone|
+--------+------+----+-----+-----+-------+----------+-----+
|       0|     3|22.0|    1|    0|   7.25|      true|false|
|       1|     1|38.0|    1|    0|71.2833|     false|false|
|       1|     3|26.0|    0|    0|  7.925|     false| true|
|       1|     1|35.0|    1|    0|   53.1|     false|false|
|       0|     3|35.0|    0|    0|   8.05|      true| true|
+--------+------+----+-----+-----+-------+----------+-----+
only showing top 5 rows


_pclass_: Ticket class (1st, 2nd, 3rd)  
_sibsp_: Number of siblings/spouses on board  
_parch_: Number of parents/children onboard

Cast booleans to integers  
(recall that scikit-learn was able to work with `bool` values directly, could it be because PySpark is actually a layer on top of Scala, which is a layer on top of Java, which does not have a way to automatically translate booleans to integers?)

In [160]:
from pyspark.sql.functions import array, col

titanic_df = titanic_df.withColumn('adult_male', col('adult_male').cast('int')).withColumn('alone', col('alone').cast('int'))
titanic_df.show(5)

+--------+------+----+-----+-----+-------+----------+-----+
|survived|pclass| age|sibsp|parch|   fare|adult_male|alone|
+--------+------+----+-----+-----+-------+----------+-----+
|       0|     3|22.0|    1|    0|   7.25|         1|    0|
|       1|     1|38.0|    1|    0|71.2833|         0|    0|
|       1|     3|26.0|    0|    0|  7.925|         0|    1|
|       1|     1|35.0|    1|    0|   53.1|         0|    0|
|       0|     3|35.0|    0|    0|   8.05|         1|    1|
+--------+------+----+-----+-----+-------+----------+-----+
only showing top 5 rows


Recall that in scikit-learn, the `.fit()` function takes two arguments for supervised learning algorithms `X` and `y`. Pyspark does things differently. It wants a single dataframe where the features are a single column and the target is a separate column.

In [161]:
from pyspark.ml.feature import VectorAssembler

feature_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare', 'adult_male', 'alone']

assembler = VectorAssembler(
    inputCols = feature_cols,
    outputCol = "features"
)

assembler

VectorAssembler_83e3c429b495

In [162]:
titanic_ml_df = assembler.transform(titanic_df).drop(*feature_cols)

titanic_ml_df.show(5)

+--------+--------------------+
|survived|            features|
+--------+--------------------+
|       0|[3.0,22.0,1.0,0.0...|
|       1|[1.0,38.0,1.0,0.0...|
|       1|[3.0,26.0,0.0,0.0...|
|       1|[1.0,35.0,1.0,0.0...|
|       0|[3.0,35.0,0.0,0.0...|
+--------+--------------------+
only showing top 5 rows


#### Split test/train

Surprisingly, _many_ pyspark examples in their official documentation or online blogs don't do a test/train split!

In [163]:
(train_data, test_data) = titanic_ml_df.randomSplit([0.8, 0.2], seed=42)

train_data.count(), test_data.count()

(576, 138)

#### Pyspark also has the concept of pipelines

(although we don't use it here)

```python
pipeline = Pipeline(stages=[labelIndexer, featureIndexer, decision_tree_classifier])
```

Example taken from https://spark.apache.org/docs/latest/ml-classification-regression.html#logistic-regression

#### Train the model

In [164]:
from pyspark.ml.classification import LogisticRegression

model = LogisticRegression(featuresCol="features", labelCol="survived").fit(train_data)

model

LogisticRegressionModel: uid=LogisticRegression_cd58834cdb79, numClasses=2, numFeatures=7

#### Run the model

In [165]:
predictions = model.transform(test_data)

In [166]:
predictions.show(5)

+--------+--------------------+--------------------+--------------------+----------+
|survived|            features|       rawPrediction|         probability|prediction|
+--------+--------------------+--------------------+--------------------+----------+
|       0|[1.0,40.0,0.0,0.0...|[0.40994914718123...|[0.60107568523371...|       0.0|
|       0|[2.0,27.0,1.0,0.0...|[-1.4412287953437...|[0.19135513468419...|       1.0|
|       0|[2.0,66.0,0.0,0.0...|[2.25392606361298...|[0.90498865168327...|       0.0|
|       0|[3.0,14.0,0.0,0.0...|[-0.8104027921348...|[0.30780466977824...|       1.0|
|       0|[3.0,22.0,1.0,0.0...|[2.34468866380066...|[0.91251112732004...|       0.0|
+--------+--------------------+--------------------+--------------------+----------+
only showing top 5 rows


#### Evaluate the model

In [167]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics

In [ ]:
evaluator = BinaryClassificationEvaluator(labelCol="survived", metricName="areaUnderPR")

eval_metric = evaluator.evaluate(predictions)
eval_metric

0.9364186888261462

25/12/02 20:15:04 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:307)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.util.Utils$.logUncaughtExceptions(Utils.scala:1937